In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn import metrics
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import  learning_curve, validation_curve ,  train_test_split
from sklearn.metrics import (RocCurveDisplay, PrecisionRecallDisplay,
                             ConfusionMatrixDisplay,
                             mean_squared_error)
import pickle

In [ ]:
data=pd.read_csv(r"train (1).csv")

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
g_labels = ['Male', 'Female']
c_labels = ['No', 'Yes']
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
fig.add_trace(go.Pie(labels=g_labels, values=data['gender'].value_counts(), name="Gender"),
              1, 1)
fig.add_trace(go.Pie(labels=c_labels, values=data['Class/ASD'].value_counts(), name="Class/ASD"),
              1, 2)


fig.update_traces(hole=.4, hoverinfo="label+percent+name", textfont_size=16)

fig.update_layout(
    title_text="Gender and Class/ASD Distributions",

    annotations=[dict(text='Gender', x=0.19, y=0.5, font_size=20, showarrow=False),
                 dict(text='Class/ASD', x=0.82, y=0.5, font_size=18, showarrow=False)])
fig.show()

In [ ]:

import plotly.express as px

# Age distribution
plt.figure(figsize=(8,5))
sns.histplot(data['age'], kde=True, bins=20, color="skyblue")
plt.title("Age Distribution")
plt.show()

In [ ]:

# Gender distribution
fig = px.histogram(data, x="gender", color="Class/ASD", barmode="group",
                   title="Gender Distribution by ASD Class")
fig.show()

In [ ]:


# Ethnicity distribution
ethnicity_count = data['ethnicity'].value_counts().head(10)
plt.figure(figsize=(8,5))
sns.barplot(x=ethnicity_count.values, y=ethnicity_count.index, palette="viridis")
plt.title("Top 10 Ethnicities")
plt.show()

In [ ]:

# Country of residence
fig = px.choropleth(data, locations="contry_of_res", locationmode="country names",
                    color="Class/ASD", title="ASD Cases by Country")
fig.show()


In [ ]:

# Jaundice
sns.countplot(x="jaundice", hue="Class/ASD", data=data, palette="Set2")
plt.title("Jaundice History vs ASD")
plt.show()

In [ ]:

# Autism in family
sns.countplot(x="austim", hue="Class/ASD", data=data, palette="coolwarm")
plt.title("Family Autism History vs ASD")
plt.show()

In [ ]:


# Used app before
fig = px.pie(data, names="used_app_before", title="Used App Before Distribution")
fig.show()


In [ ]:

# Relation
plt.figure(figsize=(10,5))
sns.countplot(y="relation", data=data, order=data['relation'].value_counts().index, palette="pastel")
plt.title("Relation to Test Taker")
plt.show()

In [ ]:


score_cols = [f"A{i}_Score" for i in range(1,11)]

# Heatmap
plt.figure(figsize=(10,6))
sns.heatmap(data[score_cols + ['result']].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap of Screening Scores")
plt.show()

In [ ]:

# Radar chart (avg per class)
import plotly.graph_objects as go
avg_scores = data.groupby("Class/ASD")[score_cols].mean().T
fig = go.Figure()
for label in avg_scores.columns:
    fig.add_trace(go.Scatterpolar(r=avg_scores[label].values,
                                  theta=avg_scores.index,
                                  fill="toself", name=label))
fig.update_layout(title="Average Screening Scores by ASD Class",
                  polar=dict(radialaxis=dict(visible=True)))
fig.show()


In [ ]:





# Class distribution
sns.countplot(x="Class/ASD", data=data, palette="muted")
plt.title("ASD Class Distribution")
plt.show()

# Gender × Class
sns.countplot(x="gender", hue="Class/ASD", data=data, palette="Set1")
plt.title("Gender vs ASD")
plt.show()

# Age × Class
plt.figure(figsize=(8,5))
sns.violinplot(x="Class/ASD", y="age", data=data, palette="Set2")
plt.title("Age vs ASD Class")
plt.show()

# Sunburst
fig = px.sunburst(data, path=["gender","ethnicity","Class/ASD"],
                  title="ASD Distribution by Gender & Ethnicity")
fig.show()


Data PreProcessing

In [ ]:
data=data.drop(["ID"] , axis=1)

In [ ]:
data["gender"] = data["gender"].replace({"f": 0, "m": 1})

In [ ]:
data = pd.get_dummies(data, columns=['ethnicity','jaundice',
       'austim', 'contry_of_res', 'used_app_before', 'age_desc',
       'relation'] , dtype=int)

In [ ]:
data.dtypes

In [ ]:
y=data["Class/ASD"]
x=data.drop(["Class/ASD"] , axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler=StandardScaler()
x=scaler.fit_transform(x)


In [ ]:
x_train , x_test , y_train , y_test =train_test_split(x ,y , test_size=0.2,random_state=42 , stratify=y)
print(x_train.shape , y_train.shape) , x_test , y_test

In [ ]:
modeles={"RandomForestClassifier":RandomForestClassifier( n_estimators=500,       
    max_depth=6,             
    min_samples_split=50,   
    min_samples_leaf=20,    
    max_features=0.3,       
    class_weight="balanced", 
    random_state=42) , 
         "GradientBoostingClassifier":GradientBoostingClassifier() ,
         "LogisticRegression" :LogisticRegression( penalty="l2",  
    C=1.0,            
    solver="lbfgs",    
    max_iter=100,      
    random_state=42)}

for name,a in modeles.items():
    a.fit(x_train,y_train)
    y_pre=a.predict(x_test)
    accuracy=accuracy_score(y_test, y_pre)
    print(f"{name}  accuracy score is {accuracy} ")
    RocCurveDisplay.from_estimator(a, x_test, y_test)
    plt.title(f"ROC Curve:{name}"); plt.show()
    PrecisionRecallDisplay.from_estimator(a, x_test, y_test)
    plt.title(f"Precision-Recall Curve:{name}"); plt.show()
    ConfusionMatrixDisplay.from_estimator(a, x_test, y_test)
    plt.title(f"Confusion Matrix:{name}"); plt.show()

In [ ]:
for name,a in modeles.items():
    train_sizes, train_scores, val_scores = learning_curve(
        a, x, y, cv=5, scoring='f1', train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1)
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train')
    plt.plot(train_sizes, val_scores.mean(axis=1), marker='o', label='Validation')
    plt.xlabel('Training examples'); plt.ylabel('F1'); plt.title(f'Learning Curve:{name}'); plt.legend(); plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
# Calibration Curve
for name,a in modeles.items():
    prob_pos = a.predict_proba(x_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, prob_pos, n_bins=10, strategy='uniform')
    plt.plot(mean_pred, frac_pos, marker='o'); plt.plot([0,1],[0,1],'--')
    plt.xlabel('Predicted probability'); plt.ylabel('True fraction of positives'); plt.title(f'Calibration{name}'); plt.show()